# Member 2 — Random Forest Classifier
## FIFA Player Position Prediction

## Dataset Description

- **Name:** FIFA 22 Complete Player Dataset
- **Source:** https://www.kaggle.com/datasets/stefanoleone992/fifa-22-complete-player-dataset
- **Size:** ~19,000 players, 100+ attributes
- **Task:** Predict player position category from physical and skill attributes
- **Features used:** age, height_cm, weight_kg, overall, potential, 
                     pace, shooting, passing, dribbling, defending, physic
- **Target:** position_category
- **Context:** FIFA 22 is a football simulation game. Player 
               statistics reflect real-world player abilities.

### Algorithm — Random Forest Classifier
- **n_estimators:** 100 trees
- **max_depth:** 10
- **Accuracy achieved:** 81.01%

---
**Dataset:** FIFA 22 Players
**Task:** Predict if a player is ATTACKER, MIDFIELDER, or DEFENDER

---



## 1. Introduction to Random Forest

A **Random Forest** is a collection of many Decision Trees working together.

Think of it like asking 100 different football experts to predict a player's position. Each expert (tree) looks at slightly different statistics. Then they all VOTE, and the most popular answer wins.

**Why Random Forest for this task?**
- More accurate than a single Decision Tree
- Very hard to overfit (memorize the training data)
- Works great with tabular sports statistics
- Shows which player attributes matter most

## 2. Import libraries

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
# Random Forest from scikit-learn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    RocCurveDisplay
)
from sklearn.preprocessing import label_binarize
print('Libraries imported!')

Libraries imported!


## 3. Load the preprocessed data

In [ ]:
X_train = np.load('X_train.npy')
X_test  = np.load('X_test.npy')
y_train = np.load('y_train.npy')
y_test  = np.load('y_test.npy')
class_names = np.load('class_names.npy', allow_pickle=True)
print('Data loaded!')
print(f'Training: {len(X_train)} | Testing: {len(X_test)} | Classes: {list(class_names)}')

Data loaded!
Training: 15345 | Testing: 3422 | Classes: ['ATTACKER', 'DEFENDER', 'MIDFIELDER']


## 4. Train the Random Forest model

In [ ]:
# Create Random Forest with 100 trees
# n_estimators=100 means 100 decision trees
# n_jobs=-1 means use all CPU cores (faster training)
rf_model = RandomForestClassifier(
    n_estimators=100,   # Number of trees
    max_depth=10,       # Max depth of each tree
    random_state=42,    # For reproducibility
    n_jobs=-1           # Use all CPU cores
)
# Train the model
print('Training Random Forest with 100 trees... (may take 10-30 seconds)')
rf_model.fit(X_train, y_train)
print('Training complete!')

## 5. Make predictions and evaluate

In [ ]:
y_pred = rf_model.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
print(f'=== RANDOM FOREST RESULTS ===')
print(f'Accuracy: {accuracy * 100:.2f}%')
print()
print('Detailed Report:')
print(classification_report(y_test, y_pred, target_names=class_names))

## 6. Evaluation Metrics 

In [ ]:
# ── 1. Precision, Recall & F1-Score ──────────────────────────────────────────
precision_macro   = precision_score(y_test, y_pred, average='macro',    zero_division=0)
precision_weighted= precision_score(y_test, y_pred, average='weighted', zero_division=0)
recall_macro      = recall_score   (y_test, y_pred, average='macro',    zero_division=0)
recall_weighted   = recall_score   (y_test, y_pred, average='weighted', zero_division=0)
f1_macro          = f1_score       (y_test, y_pred, average='macro',    zero_division=0)
f1_weighted       = f1_score       (y_test, y_pred, average='weighted', zero_division=0)

print('=== PRECISION, RECALL & F1-SCORE ===')
print(f'  Precision  (macro)    : {precision_macro:.4f}')
print(f'  Precision  (weighted) : {precision_weighted:.4f}')
print(f'  Recall     (macro)    : {recall_macro:.4f}')
print(f'  Recall     (weighted) : {recall_weighted:.4f}')
print(f'  F1-Score   (macro)    : {f1_macro:.4f}')
print(f'  F1-Score   (weighted) : {f1_weighted:.4f}')
print()

# Per-class breakdown
precision_per = precision_score(y_test, y_pred, average=None, zero_division=0)
recall_per    = recall_score   (y_test, y_pred, average=None, zero_division=0)
f1_per        = f1_score       (y_test, y_pred, average=None, zero_division=0)

print(f'{"Class":<20} {"Precision":>10} {"Recall":>10} {"F1-Score":>10}')
print('-' * 52)
for i, cls in enumerate(class_names):
    print(f'{cls:<20} {precision_per[i]:>10.4f} {recall_per[i]:>10.4f} {f1_per[i]:>10.4f}')

In [ ]:
# ── 2. ROC-AUC (One-vs-Rest for multi-class) ──────────────────────────────────
# Get probability scores for each class
y_prob = rf_model.predict_proba(X_test)

# Binarize labels for one-vs-rest ROC
classes = np.unique(y_train)
y_test_bin = label_binarize(y_test, classes=classes)

roc_auc_macro    = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='macro')
roc_auc_weighted = roc_auc_score(y_test_bin, y_prob, multi_class='ovr', average='weighted')

print('=== ROC-AUC SCORE (One-vs-Rest) ===')
print(f'  ROC-AUC (macro)    : {roc_auc_macro:.4f}')
print(f'  ROC-AUC (weighted) : {roc_auc_weighted:.4f}')

# Plot ROC curve for each class
from sklearn.metrics import roc_curve, auc

plt.figure(figsize=(10, 7))
colors = plt.cm.Set2(np.linspace(0, 1, len(classes)))

for i, (cls_idx, color) in enumerate(zip(classes, colors)):
    fpr, tpr, _ = roc_curve(y_test_bin[:, i], y_prob[:, i])
    roc_auc_cls = auc(fpr, tpr)
    plt.plot(fpr, tpr, color=color, lw=2,
             label=f'{class_names[i]} (AUC = {roc_auc_cls:.2f})')

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random Classifier')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Random Forest — ROC Curve (One-vs-Rest per Class)')
plt.legend(loc='lower right', fontsize=8)
plt.tight_layout()
plt.savefig('member2_roc_curve.png')
plt.show()
print('ROC curve saved!')

In [ ]:
# ── 3. Confusion Matrix ───────────────────────────────────────────────────────
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names,
            yticklabels=class_names)
plt.title('Random Forest — Confusion Matrix')
plt.ylabel('Actual Position')
plt.xlabel('Predicted Position')
plt.tight_layout()
plt.savefig('member2_confusion_matrix.png')
plt.show()
print('Confusion matrix saved!')

# TP / FP / TN / FN breakdown per class
print()
print(f'{"Class":<20} {"TP":>6} {"FP":>6} {"TN":>6} {"FN":>6}')
print('-' * 44)
for i, cls in enumerate(class_names):
    TP = cm[i, i]
    FP = cm[:, i].sum() - TP
    FN = cm[i, :].sum() - TP
    TN = cm.sum() - TP - FP - FN
    print(f'{cls:<20} {TP:>6} {FP:>6} {TN:>6} {FN:>6}')